# Effective Software Testing - Fuzzing Assignment
Both the description and the solution of this assignment will take place in this interactive jupyter notebook. Your deliverable should contain
- a `fuzzing_assignment.ipynb`, which will be the edited version of this file that will contain your solution.
- an `assignment_utils.py` that we provide you and holds helper classes for the assignment. Generally speaking, you should not edit this file, but see below for details.

The assignment *can* be solved only by filling in the cells where there is the comment "# Your code here", **but**, many solutions exist, and it is possible that some of them involve editing other cells of even the `assignment_utils.py`. The comments ("# Your code here") are **optional**, and they just indicate an exemplary solution that we will make available to you after the deadline. Feel free to edit any other cell, file, or even install new packages (in which case you must submit them in a comment at the beginning). Finally, any comments, documentation, or other remarks in natural language should take place inside this notebook, in a markdown cell.

For any question about the assignment, feel free to reach out to `alexandridis@ifi.uzh.ch`.

Read below for more, and good luck!

# Installation
Python 3.13 is required for this assignment.
To install the required packages, create a new `pip` virtual environment and install the required libraries by running
```
pip install -r requirements.txt
```

If you encounter issues with the python version or the installation of requirements.txt, please reach out to the ticketing system.

# Setup
The first part of the notebook sets up the environment by implementing classes/functions that were covered during the lecture.

In [19]:
from ast import literal_eval
import random
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import pickle
import hashlib
from typing import List, Set, Any

In [20]:
from assignment_utils import Fuzzer, Runner, FunctionCoverageRunner 

Remember the `urlparse()` function from the lecture? We will use a similar function in this assignment. Namely the function `literal_eval()` from the `ast` package, that parses a string representation of a python variable into an object.

In [21]:
test_string = '[1, 2, 3, False]'
literal_eval(test_string)

[1, 2, 3, False]

Note that the function throws an exception when provided with malformed input, so it is ready to be used for fuzzing.

Now, we will copy the code of the second live coding session we did in during the lecture. The code below implements the mutation part of a mutation-based blackbox fuzzer.

In [22]:
def mutate(s, num_of_mutations = 1):
    mutators = [delete_random, insert_random, flip_random]

    for i in range(num_of_mutations):
        mutator_ind = random.randint(0, 2)
        selected_mutator = mutators[mutator_ind]
        s = selected_mutator(s)

    return s

In [23]:
def delete_random(s):
    pos_to_delete = random.randint(0, len(s) - 1)
    return s[:pos_to_delete-1] + s[pos_to_delete:]

In [24]:
def insert_random(s):
    pos_to_insert = random.randint(0, len(s)-1)
    char_to_insert = chr(random.randint(32, 126)) # check here for chr table: https://dev.to/elkogit/python-ord-and-chr-3pin
    return s[:pos_to_insert-1] + char_to_insert + s[pos_to_insert-1:]

In [25]:
def flip_random(s):
    pos_to_flip  = random.randint(0, len(s)-1)
    byte_to_flip = ord(s[pos_to_flip]) # char --> byte
    bit_to_flip  = 1 << random.randint(0, 6) # 0010000 or 100000
    flipped_char = chr(byte_to_flip ^ bit_to_flip)

    return s[:pos_to_flip] + flipped_char + s[pos_to_flip+1:]

To put the function `mutate()` in use, we will implement the mutation-based blackbox fuzzer in the `MutationBlackboxFuzzer` class as following. The base class `Fuzzer` is just a placeholder that you can find in `assignment_utils.py`.

In [26]:
class MutationBlackboxFuzzer(Fuzzer):
    """Base class for mutational fuzzing"""

    def __init__(self, seed: List[str],
                 min_mutations: int = 2,
                 max_mutations: int = 10) -> None:
        """Constructor.
        `seed` - a list of (input) strings to mutate.
        `min_mutations` - the minimum number of mutations to apply.
        `max_mutations` - the maximum number of mutations to apply.
        """
        self.seed = seed
        self.min_mutations = min_mutations
        self.max_mutations = max_mutations
        self.reset()

    def reset(self) -> None:
        """Set population to initial seed.
        To be overloaded in subclasses."""
        self.population = self.seed
        self.seed_index = 0

    def mutate(self, inp: str) -> str:
        return mutate(inp)

    def create_candidate(self) -> str:
        """Create a new candidate by mutating a population member"""
        candidate = random.choice(self.population)
        trials = random.randint(self.min_mutations, self.max_mutations)
        for i in range(trials):
            candidate = self.mutate(candidate)
        return candidate

    def fuzz(self) -> str:
        # If we did not return all the seeds, return the next seed. Only after we run 
        # out of seeds, we return a mutated version of the seed
        if self.seed_index < len(self.seed):
            # Still seeding
            self.inp = self.seed[self.seed_index]
            self.seed_index += 1
        else:
            # Mutating
            self.inp = self.create_candidate()
        return self.inp

Remember the `Runner` class that was used in the lecture to measure the coverage achieved by a specific input. Instead of calling
```python
result = literal_eval("[1, 2, 3, false]")
```

you can instead do
```python
http_runner = FunctionCoverageRunner(literal_eval) # define the function here
result, isCrash = http_runner.run('[1, 2, 3, False]') # define the input here
```
which also gives you access to the coverage achieved:
```
list(http_runner.coverage())
```

In [27]:
http_runner = FunctionCoverageRunner(literal_eval) # Only initialize it once per function!
http_runner.run('[1, 2, 3, False]')

len(list(http_runner.coverage()))

22

Putting it all together, we can now run our fuzzer with a single seed for 1000 trials and measure the number of covered lines  like this

In [28]:
seed_input = '[1, 2, 3, False]'
mutation_fuzzer = MutationBlackboxFuzzer(seed=[seed_input])

mutation_fuzzer.runs(http_runner, trials=1000)
print(mutation_fuzzer.population)

print("%d lines covered" % len(http_runner.coverage()))

['[1, 2, 3, False]']
37 lines covered


# Task 01 - Investigating the Effect of Seed Input Complexity
In the above run, our fuzzer covers 37 lines using only one starting seed (`'[1, 2, 3, False]'`). Your first task is to investigate the effect of the complexity of starting seed inputs in fuzzer performance. We remind that fuzzer performance is usually measure by the achieved coverage of the fuzzer (higher coverage => higher probability of exposing bugs => better performance). As a function to test, use the `lliteral_eval()` function from the `ast` library.

Compare the coverage of the `MutationBlackboxFuzzer` with five simple seeds vs five complex seeds. Seeds should all have similar complexity (for our example here, have a similar number of elements and depth of nested ones). For simple seeds, use around 3-5 top level keys, with mostly numerical values (integers and floats), while for complex seeds use nested variables of up to 3 levels deep, mixing string, boolean and numerical values.

The choice of the ten seeds is up to you, you can either craft them by hand, or write a small script that randomly generates them.

Comparing the performance of two fuzzers (or the same fuzzer with different configurations in our case) is a nuanced topic due to the inherent randomness of fuzzing. For example, if we run the `MutationBlackboxFuzzer` with one seed and cover 40 lines, and then run it with ten seeds and cover 42 lines, can we claim that the more starting seeds the better? Probably not.
To make claims based on empirical data, we have to run statistical tests. In fuzzing, typically each fuzzer configuration is ran for 30 runs of 10,000 trials (i.e., you run the above cell 30 times), yielding an array of 30 coverage values for each fuzzer. The two arrays are then compared using the Mann-Whitney U Test with the null hypothesis that the values of the first array (i.e., coverage of the first fuzzer) are greater than those of the second array. If the p-value of the test is <0.05, we reject the null hypothesis, claiming that the first fuzzer achieves less coverage than the second one (with confidence level 1-0.05 = 95%).
You can run the Mann-Whitney U Test in python with the following code:
```python
from scipy.stats import mannwhitneyu
u_stat, p_value = mannwhitneyu(c1, c2, alternative='less')
if p_value < 0.05:
    print("Values of c1 are smaller than those in c2 with confidence level 95%")
```

In [29]:
# Task 01 solution

def run_blackbox_coverage(seeds, trials=10_000):
    runner = FunctionCoverageRunner(literal_eval)
    fuzzer = MutationBlackboxFuzzer(seed=seeds)
    fuzzer.runs(runner, trials=trials)
    return len(runner.coverage())


simple_seeds = [
    "[1, 2, 3, 4]",
    "[10.5, 20.5, 30.5]",
    "{'a': 1, 'b': 2, 'c': 3}",
    "(1, 2, 3, 4)",
    "{'x': 1.0, 'y': 2.0, 'z': 3.0}"
]

complex_seeds = [
    "{'a': [1, {'b': 2}], 'c': (3, 4), 'd': True}",
    "[{'k1': [1, 2]}, {'k2': {'k3': [3.1, False]}}]",
    "{'u': {'v': {'w': 7}}, 'p': [1, 2, 3], 'q': 'txt'}",
    "[{'x': {'y': [1, 2]}}, {'z': (True, False)}, None]",
    "{'m': [{'n': 1}, {'o': [2, {'p': 3}]}], 'r': 4.5}"
]

runs = 30
simple_cov = [run_blackbox_coverage(simple_seeds) for _ in range(runs)]
complex_cov = [run_blackbox_coverage(complex_seeds) for _ in range(runs)]

u_stat, p_value = mannwhitneyu(simple_cov, complex_cov, alternative='less')

print("Simple seeds coverage:", simple_cov)
print("Complex seeds coverage:", complex_cov)
print(f"Simple mean={sum(simple_cov)/len(simple_cov):.2f}")
print(f"Complex mean={sum(complex_cov)/len(complex_cov):.2f}")
print(f"Mann-Whitney U={u_stat}, p-value={p_value:.6f}")

if p_value < 0.05:
    print("Simple seeds achieve significantly lower coverage than complex seeds (95% confidence).")
else:
    print("No statistically significant evidence that simple seeds are worse.")

<unknown>:1: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\>" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\>"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\ " is an i

Simple seeds coverage: [12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 37, 12, 12, 12, 12, 12, 12, 12, 12]
Complex seeds coverage: [12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]
Simple mean=12.83
Complex mean=12.00
Mann-Whitney U=465.0, p-value=0.849276
No statistically significant evidence that simple seeds are worse.


<unknown>:1: SyntaxWarning: "\k" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\k"? A raw string is also an option.


# Task 02: Investigating the Effect of More Sophisticated Mutations
The mutation function we implemented above (and in the lecture) is:
```python
def mutate(s, num_of_mutations = 1):
    mutators = [delete_random, insert_random, flip_random]

    for i in range(num_of_mutations):
        mutator_ind = random.randint(0, 2)
        selected_mutator = mutators[mutator_ind]
        s = selected_mutator(s)

    return s
```

Your task is to implement a more sophisticated mutation strategy by:

1. Extending the `mutate(s, num_of_mutations = 1)` function above to support three more mutators
    - Swap two random bytes
    - Bitwise XOR with a random byte on a random position
    - Randomly rotate the bits of a random byte by a random number of positions
2. Extending the `mutate(self, s)` method of `MutationBlackboxFuzzer` so that with probability `p` it calls the `mutate(s, num_of_mutations = 1)` function above and with probability `1-p` mutates the input using **chunk deletion and duplication**. In this case, a random chunk of the substring is deleted, and replaced with a duplicated value from another part of the input. Note that the duplicated chunk is repeated once.

Compare the performance of your new fuzzer with the previous fuzzer over 30 runs of 5000 trials each. Again, use the `literal_eval()` function as the function to be tested. Use a single starting seed `s=["[{'a': [1, 2]}, {'b': (3, 4)}, None, True, False]"]`. What can you claim about the effect of more complex mutations?

Note: For more information about the exact mutations used in the most popular fuzzer, AFL, you can have a look at this blog by the AFL creator: https://lcamtuf.blogspot.com/2014/08/binary-fuzzing-strategies-what-works.html

In [30]:
# Task 02 solution

def safe_delete_random(s):
    if not s:
        return s
    pos = random.randint(0, len(s) - 1)
    return s[:pos] + s[pos + 1:]


def safe_insert_random(s):
    pos = random.randint(0, len(s))
    ch = chr(random.randint(32, 126))
    return s[:pos] + ch + s[pos:]


def safe_flip_random(s):
    if not s:
        return s
    pos = random.randint(0, len(s) - 1)
    byte_to_flip = ord(s[pos])
    bit_to_flip = 1 << random.randint(0, 6)
    flipped = chr(byte_to_flip ^ bit_to_flip)
    return s[:pos] + flipped + s[pos + 1:]


def swap_random(s):
    if len(s) < 2:
        return s
    i, j = random.sample(range(len(s)), 2)
    chars = list(s)
    chars[i], chars[j] = chars[j], chars[i]
    return ''.join(chars)


def xor_random(s):
    if not s:
        return s
    pos = random.randint(0, len(s) - 1)
    x = random.randint(1, 255)
    new_c = chr(ord(s[pos]) ^ x)
    return s[:pos] + new_c + s[pos + 1:]


def rotate_bits_random(s):
    if not s:
        return s
    pos = random.randint(0, len(s) - 1)
    b = ord(s[pos]) & 0xFF
    r = random.randint(1, 7)
    rb = ((b << r) | (b >> (8 - r))) & 0xFF
    return s[:pos] + chr(rb) + s[pos + 1:]


def mutate_extended(s, num_of_mutations=1):
    mutators = [safe_delete_random, safe_insert_random, safe_flip_random, swap_random, xor_random, rotate_bits_random]
    for _ in range(num_of_mutations):
        s = random.choice(mutators)(s)
    return s


def chunk_delete_duplicate(s):
    if len(s) < 2:
        return safe_insert_random(s)

    a = random.randint(0, len(s) - 1)
    b = random.randint(a, len(s) - 1)
    reduced = s[:a] + s[b + 1:]
    if not reduced:
        return safe_insert_random(reduced)

    c = random.randint(0, len(reduced) - 1)
    d = random.randint(c, len(reduced) - 1)
    chunk = reduced[c:d + 1]
    insert_at = random.randint(0, len(reduced))
    return reduced[:insert_at] + chunk + chunk + reduced[insert_at:]


class SophisticatedMutationBlackboxFuzzer(MutationBlackboxFuzzer):
    def __init__(self, seed, p=0.7, min_mutations=2, max_mutations=10):
        super().__init__(seed, min_mutations=min_mutations, max_mutations=max_mutations)
        self.p = p

    def mutate(self, inp: str) -> str:
        if random.random() < self.p:
            return mutate_extended(inp, num_of_mutations=1)
        return chunk_delete_duplicate(inp)


def run_fuzzer_coverage(fuzzer_cls, seed, trials):
    runner = FunctionCoverageRunner(literal_eval)
    fuzzer = fuzzer_cls(seed=seed)
    fuzzer.runs(runner, trials=trials)
    return len(runner.coverage())


seed = ["[{'a': [1, 2]}, {'b': (3, 4)}, None, True, False]"]
runs = 30
trials = 5000

baseline_cov = [run_fuzzer_coverage(MutationBlackboxFuzzer, seed, trials) for _ in range(runs)]
advanced_cov = [run_fuzzer_coverage(SophisticatedMutationBlackboxFuzzer, seed, trials) for _ in range(runs)]

u_stat, p_value = mannwhitneyu(baseline_cov, advanced_cov, alternative='less')

print("Baseline coverage:", baseline_cov)
print("Advanced coverage:", advanced_cov)
print(f"Baseline mean={sum(baseline_cov)/len(baseline_cov):.2f}")
print(f"Advanced mean={sum(advanced_cov)/len(advanced_cov):.2f}")
print(f"Mann-Whitney U={u_stat}, p-value={p_value:.6f}")

if p_value < 0.05:
    print("Baseline is significantly worse than advanced mutations (95% confidence).")
else:
    print("No statistically significant evidence that advanced mutations are better.")

<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\:" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\:"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an i

Baseline coverage: [12, 12, 12, 12, 12, 37, 12, 12, 12, 12, 12, 29, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 40]
Advanced coverage: [12, 12, 12, 12, 12, 12, 12, 12, 12, 41, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]
Baseline mean=14.33
Advanced mean=12.97
Mann-Whitney U=478.5, p-value=0.839275
No statistically significant evidence that advanced mutations are better.


# Task 03: Compare With Mutation-Based Greybox Fuzzer
To improve on the blackbox fuzzer in the lecture, we implemented the greybox fuzzer below, that dynamically adds a mutated input to the seeds queue if that input covers a new path. 

Your task is to compare the performance of the (already implemented) greybox fuzzer with that of the blackbox fuzzer over 30 runs of 5000 trials each.
Use the same single starting seed as the previous experiment.

What can you claim about the performance of they greybox fuzzer?

In [31]:
class MutationGreyboxFuzzer(MutationBlackboxFuzzer):
    """Fuzz with mutated inputs based on coverage"""

    def reset(self) -> None:
        super().reset()
        self.coverages_seen: Set[frozenset] = set()
        # Now empty; we fill this with seed in the first fuzz runs
        self.population = []

    def run(self, runner: FunctionCoverageRunner) -> Any:
        """Run function(inp) while tracking coverage.
           If we reach new coverage,
           add inp to population and its coverage to population_coverage
        """
        result, outcome = super().run(runner)
        new_coverage = frozenset(runner.coverage())
        if outcome == Runner.PASS and new_coverage not in self.coverages_seen:
            # We have new coverage
            self.population.append(self.inp)
            self.coverages_seen.add(new_coverage)

        return result

In [32]:
# Task 03 solution

def run_blackbox_literal_eval(seed, trials):
    runner = FunctionCoverageRunner(literal_eval)
    fuzzer = MutationBlackboxFuzzer(seed=seed)
    fuzzer.runs(runner, trials=trials)
    return len(runner.coverage())


def run_greybox_literal_eval(seed, trials):
    runner = FunctionCoverageRunner(literal_eval)
    fuzzer = MutationGreyboxFuzzer(seed=seed)
    for _ in range(trials):
        fuzzer.run(runner)
    return len(runner.coverage())


seed = ["[{'a': [1, 2]}, {'b': (3, 4)}, None, True, False]"]
runs = 30
trials = 5000

blackbox_cov = [run_blackbox_literal_eval(seed, trials) for _ in range(runs)]
greybox_cov = [run_greybox_literal_eval(seed, trials) for _ in range(runs)]

u_stat, p_value = mannwhitneyu(blackbox_cov, greybox_cov, alternative='less')

print("Blackbox coverage:", blackbox_cov)
print("Greybox coverage:", greybox_cov)
print(f"Blackbox mean={sum(blackbox_cov)/len(blackbox_cov):.2f}")
print(f"Greybox mean={sum(greybox_cov)/len(greybox_cov):.2f}")
print(f"Mann-Whitney U={u_stat}, p-value={p_value:.6f}")

if p_value < 0.05:
    print("Blackbox is significantly worse than greybox (95% confidence).")
else:
    print("No statistically significant evidence that greybox is better.")

<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\<" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\<"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an i

Blackbox coverage: [12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 41, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 41, 12, 12, 12, 12]
Greybox coverage: [12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 19, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]
Blackbox mean=13.93
Greybox mean=12.23
Mann-Whitney U=466.0, p-value=0.740842
No statistically significant evidence that greybox is better.


# Task 04: Boosted Greybox Fuzzer
Towards the end of the guest lecture, we discussed about boosting the performance of greybox fuzzers by prioritizing seeds that exercise more rare paths. Your task is to implement this into a class named `BoostedMutationGreyboxFuzzer` and compare its performance against the `MutationGreyboxFuzzer`.

Specifically, you must calculate the coverage achieved for each input you feed into the program-under-test, and keep track of how frequently a specific coverage is triggered (hint: you may find the function `getPathID()` below useful for this). 

Then, when selecting the next seed to mutate, you must select with probability reversly proportional to the frequency of the coverage of each seed. As a result, seeds that exercise rarer paths will be selected more frequently.

Compare the performance of your new `BoostedMutationGreyboxFuzzer` with the base `MutationGreyboxFuzzer` in two target programs:
1. On the `literal_eval()` program using 30 runs of 5,000 trials each and the same single starting seed as above.
2. On the `crashme()` program found below, 30 runs of using 10,000 trials and a single starting seed `s=["good"]`.

In [33]:
def getPathID(coverage: Any) -> str:
    """Returns a unique hash for the covered statements"""
    pickled = pickle.dumps(sorted(coverage))
    return hashlib.md5(pickled).hexdigest()

In [34]:
def crashme(s: str) -> None:
    if len(s) > 0 and s[0] == 'z':
        if len(s) > 1 and s[1] == 'e':
            if len(s) > 2 and s[2] == 's':
                if len(s) > 3 and s[3] == 't':
                    raise Exception()

In [35]:
class BoostedMutationGreyboxFuzzer(MutationBlackboxFuzzer):
    """Fuzz with mutated inputs based on coverage frequency"""

    def reset(self) -> None:
        super().reset()
        self.coverages_seen: Set[frozenset] = set()
        # Now empty; we fill this with seed in the first fuzz run
        self.population = []
        self.seed_path_id = {}
        self.path_frequency = {}

    def run(self, runner: FunctionCoverageRunner) -> Any:
        result, outcome = super().run(runner)
        cov = frozenset(runner.coverage())
        if outcome == Runner.PASS:
            path_id = getPathID(cov)
            if cov not in self.coverages_seen:
                self.population.append(self.inp)
                self.coverages_seen.add(cov)
                self.seed_path_id[self.inp] = path_id
            # update path frequency for this run
            self.path_frequency[path_id] = self.path_frequency.get(path_id, 0) + 1
        return result

    def create_candidate(self) -> str:
        if not self.population:
            return random.choice(self.seed)

        # Weight inversely proportional to frequency -> rare paths selected more often
        weights = []
        for s in self.population:
            pid = self.seed_path_id.get(s)
            freq = self.path_frequency.get(pid, 1)
            weights.append(1.0 / freq)

        candidate = random.choices(self.population, weights=weights, k=1)[0]
        trials = random.randint(self.min_mutations, self.max_mutations)
        for _ in range(trials):
            candidate = self.mutate(candidate)
        return candidate

In [36]:
# Task 04 solution

def run_greybox_target(function, seed, trials):
    runner = FunctionCoverageRunner(function)
    fuzzer = MutationGreyboxFuzzer(seed=seed)
    for _ in range(trials):
        fuzzer.run(runner)
    return len(runner.coverage())


def run_boosted_greybox_target(function, seed, trials):
    runner = FunctionCoverageRunner(function)
    fuzzer = BoostedMutationGreyboxFuzzer(seed=seed)
    for _ in range(trials):
        fuzzer.run(runner)
    return len(runner.coverage())


def run_crash_count_greybox(fuzzer_cls, function, seed, trials):
    runner = FunctionCoverageRunner(function)
    fuzzer = fuzzer_cls(seed=seed)
    crashes = 0

    for _ in range(trials):
        inp = fuzzer.fuzz()
        _, outcome = runner.run(inp)
        cov = frozenset(runner.coverage())

        if fuzzer_cls is MutationGreyboxFuzzer:
            if outcome == Runner.PASS and cov not in fuzzer.coverages_seen:
                fuzzer.population.append(fuzzer.inp)
                fuzzer.coverages_seen.add(cov)
        else:
            if outcome == Runner.PASS:
                path_id = getPathID(cov)
                if cov not in fuzzer.coverages_seen:
                    fuzzer.population.append(fuzzer.inp)
                    fuzzer.coverages_seen.add(cov)
                    fuzzer.seed_path_id[fuzzer.inp] = path_id
                fuzzer.path_frequency[path_id] = fuzzer.path_frequency.get(path_id, 0) + 1

        if outcome == Runner.FAIL:
            crashes += 1

    return crashes


runs = 30

# 1) literal_eval comparison
seed_literal = ["[{'a': [1, 2]}, {'b': (3, 4)}, None, True, False]"]
trials_literal = 5000

base_literal_cov = [run_greybox_target(literal_eval, seed_literal, trials_literal) for _ in range(runs)]
boost_literal_cov = [run_boosted_greybox_target(literal_eval, seed_literal, trials_literal) for _ in range(runs)]

u_stat_lit, p_value_lit = mannwhitneyu(base_literal_cov, boost_literal_cov, alternative='less')

print("literal_eval base greybox coverage:", base_literal_cov)
print("literal_eval boosted greybox coverage:", boost_literal_cov)
print(f"literal_eval base mean={sum(base_literal_cov)/len(base_literal_cov):.2f}")
print(f"literal_eval boosted mean={sum(boost_literal_cov)/len(boost_literal_cov):.2f}")
print(f"literal_eval Mann-Whitney U={u_stat_lit}, p-value={p_value_lit:.6f}")

if p_value_lit < 0.05:
    print("On literal_eval, base greybox is significantly worse than boosted (95% confidence).")
else:
    print("On literal_eval, no statistically significant advantage for boosted greybox.")

# 2) crashme comparison
seed_crash = ["good"]
trials_crash = 10_000

base_crash_cov = [run_greybox_target(crashme, seed_crash, trials_crash) for _ in range(runs)]
boost_crash_cov = [run_boosted_greybox_target(crashme, seed_crash, trials_crash) for _ in range(runs)]

base_crash_hits = [run_crash_count_greybox(MutationGreyboxFuzzer, crashme, seed_crash, trials_crash) for _ in range(runs)]
boost_crash_hits = [run_crash_count_greybox(BoostedMutationGreyboxFuzzer, crashme, seed_crash, trials_crash) for _ in range(runs)]

u_stat_cov, p_value_cov = mannwhitneyu(base_crash_cov, boost_crash_cov, alternative='less')
u_stat_hit, p_value_hit = mannwhitneyu(base_crash_hits, boost_crash_hits, alternative='less')

print("\ncrashme base greybox coverage:", base_crash_cov)
print("crashme boosted greybox coverage:", boost_crash_cov)
print(f"crashme coverage p-value={p_value_cov:.6f}")

print("crashme base crash hits:", base_crash_hits)
print("crashme boosted crash hits:", boost_crash_hits)
print(f"crashme crash-hit p-value={p_value_hit:.6f}")

<unknown>:1: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\," is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\,"? A raw string is also an option.
<unknown>:1: SyntaxWarning: "\{" is an i

literal_eval base greybox coverage: [12, 12, 12, 12, 12, 12, 41, 12, 12, 12, 41, 12, 12, 12, 12, 12, 29, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]
literal_eval boosted greybox coverage: [12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 40, 12, 12, 12, 12, 40, 12, 12, 12, 12, 12, 12]
literal_eval base mean=14.50
literal_eval boosted mean=13.87
literal_eval Mann-Whitney U=466.0, p-value=0.694608
On literal_eval, no statistically significant advantage for boosted greybox.

crashme base greybox coverage: [2, 2, 2, 2, 2, 3, 2, 2, 2, 4, 2, 3, 2, 2, 2, 2, 4, 3, 2, 3, 2, 2, 3, 2, 2, 5, 3, 3, 2, 3]
crashme boosted greybox coverage: [3, 3, 5, 3, 3, 2, 3, 2, 2, 4, 2, 2, 4, 5, 2, 2, 5, 3, 4, 2, 2, 2, 2, 4, 4, 3, 2, 3, 2, 4]
crashme coverage p-value=0.032495
crashme base crash hits: [0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 2, 1, 1, 0, 0, 1, 4, 1, 0, 0, 2, 0, 2, 0, 0, 0, 41, 3, 0, 0]
crashme boosted crash hits: [2, 4, 0, 2, 0, 17, 10, 7, 0, 2, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 4, 